# Diseño experimental: 140 modelos con validación anidada

## El pipeline combinatorio

Cada modelo base se combina con cada técnica de balanceo (solo en clasificación) y cada combinación se ajusta con cada método de optimización. El total es un **producto**, no una suma:

$$\underbrace{7}_{\text{modelos}} \times \underbrace{4}_{\text{balanceo}} \times \underbrace{4}_{\text{optimizadores}} = 112 \text{ (clasificación)}, \qquad 7 \times 4 = 28 \text{ (regresión)}, \qquad 112 + 28 = 140.$$

Como cada combinación se evalúa en 3 pliegues externos, el experimento tiene **420 tareas** de optimización, más 140 reajustes finales. Todo se programa con bucles sobre funciones parametrizadas (modelo → balanceo → optimizador) y se ejecuta en paralelo con `joblib` (10 procesos).

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path[:0] = [os.path.abspath(".."), os.path.abspath("../.."), os.path.abspath("../../entregable1")]  # e3, src y e1
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import contextlib, io  # Silenciar un diagnóstico de joblib en Windows
from joblib.externals.loky import cpu_count  # Conteo de núcleos
with contextlib.redirect_stderr(io.StringIO()):  # joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché)
import pickle  # Lectura de resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Paneles
from IPython.display import Markdown, display  # Texto enriquecido
from e1.figuras import mostrar, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO  # Estilo de figuras (tema claro del informe)
from e3 import config as c3, datos as dd, modelos as mo, optimizadores as op, experimento as ex  # Código del entregable
c3.fijar_semilla()  # Semilla global (42)
pd.set_option("display.width", 170)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
COLORES = {"Grid Search": GRIS, "Random Search": AMBAR, "Bayesiana (Optuna TPE)": AZUL, "Genética (DEAP)": VERDE, "Successive Halving": MORADO}  # Color por optimizador
pd.DataFrame({"Etapa 1 · modelos (clasificación)": mo.MODELOS_CLF, "Etapa 1 · modelos (regresión)": mo.MODELOS_REG,
              "Etapa 2 · balanceo": mo.BALANCEOS + ["", "", ""], "Etapa 3 · optimizadores": list(op.METODOS) + ["", "", ""]})

,Etapa 1 · modelos (clasificación),Etapa 1 · modelos (regresión),Etapa 2 · balanceo,Etapa 3 · optimizadores
0,KNN,KNN,Sin balanceo,Grid Search
1,Naive Bayes,Ridge,SMOTE,Random Search
2,Regresión logística,Lasso,ADASYN,Bayesiana (Optuna TPE)
3,Árbol de decisión,Árbol de decisión,class_weight balanced,Genética (DEAP)
4,Random Forest,Random Forest,,
5,XGBoost,XGBoost,,
6,SVM,SVR,,


## Espacios de búsqueda

Los rangos se eligieron por la escala natural de cada hiperparámetro y por lo aprendido en el Primer Entregable; los que cambian por órdenes de magnitud ($C$, $\alpha$, tasa de aprendizaje, `var_smoothing`, vecinos, mínimo de filas por hoja) se exploran en **escala logarítmica**.

In [2]:
filas = []  # Tabla de espacios
for tarea, espacios in [("clasificación", mo.ESPACIOS_CLF), ("regresión", mo.ESPACIOS_REG)]:  # Tareas
    for modelo, esp in espacios.items():  # Modelos
        for h, spec in esp.items():  # Hiperparámetros
            rango = ", ".join(map(str, spec[1])) if spec[0] == "cat" else f"[{spec[1]:g}, {spec[2]:g}]" + (" (log)" if spec[3] else "")  # Rango legible
            filas.append({"tarea": tarea, "modelo": modelo, "hiperparámetro": h, "tipo": spec[0], "rango": rango})  # Fila
pd.DataFrame(filas)

,tarea,modelo,hiperparámetro,tipo,rango
0,clasificación,KNN,n_neighbors,int,"[5, 301] (log)"
1,clasificación,KNN,weights,cat,"uniform, distance"
2,clasificación,Naive Bayes,var_smoothing,float,"[1e-12, 0.01] (log)"
3,clasificación,Regresión logística,C,float,"[0.0001, 10] (log)"
4,clasificación,Regresión logística,penalty,cat,"l1, l2"
5,clasificación,Árbol de decisión,max_depth,int,"[2, 20]"
6,clasificación,Árbol de decisión,min_samples_leaf,int,"[5, 500] (log)"
7,clasificación,Árbol de decisión,criterion,cat,"gini, entropy"
8,clasificación,Random Forest,n_estimators,int,"[50, 150]"
9,clasificación,Random Forest,max_depth,int,"[4, 20]"


**Justificación de los rangos.**

| Modelo | Decisión | Razón |
|---|---|---|
| KNN | 5 a 301 vecinos (log), voto uniforme o por distancia | Con 20 000 filas y 10 % de brotes, pocos vecinos dan probabilidades ruidosas; el Primer Entregable eligió k ≈ 150 |
| Naive Bayes | `var_smoothing` de 1e−12 a 1e−2 (log) | Es el único hiperparámetro: estabiliza varianzas casi nulas (variables binarias y con ceros) |
| Regresión logística | C de 1e−4 a 10 (log), L1 o L2 | L1 selecciona variables entre bloques colineales; L2 reparte el peso |
| Árbol de decisión | profundidad 2–20, 5–500 filas por hoja (log), Gini o entropía | Controla la varianza de un árbol único |
| Random Forest | 50–150 árboles, profundidad 4–20, 5–200 por hoja (log), √p o 50 % de variables | Más árboles reducen varianza con rendimientos decrecientes; acotado por costo |
| XGBoost | 50–400 árboles, profundidad 2–8, tasa 0,01–0,3 (log), submuestreo 0,5–1, peso mínimo 1–50 (log) | Árboles poco profundos y tasas bajas: el régimen habitual del boosting |
| SVM / SVR | C de 1e−3 a 10 y γ de 1e−3 a 1 (log); ε de 0,01 a 0,5 en SVR | Núcleo RBF aproximado con Nyström (300 componentes) para evitar el costo cuadrático del núcleo exacto |
| Ridge / Lasso | α de 1e−3 a 1e3 (Ridge) y de 1e−4 a 1 (Lasso), log | Lasso con α grande anula todas las variables: el rango se acota |

## Balanceo de clases

| Estrategia | Cómo actúa | Implementación |
|---|---|---|
| Sin balanceo | Línea base | — |
| SMOTE | Interpola brotes sintéticos entre vecinos brote ($k = 5$) hasta igualar las clases | `imblearn` dentro del Pipeline |
| ADASYN | Como SMOTE, pero más sintéticos donde un brote tiene más vecinos no brote | `imblearn` dentro del Pipeline |
| class_weight balanced | Cada error de la clase $c$ pesa $n/(2n_c)$ | Parámetro nativo; `sample_weight` para Naive Bayes y XGBoost; corrección por la probabilidad a priori en KNN |

KNN no acepta pesos de muestra. Su equivalente es dividir la proporción de vecinos de cada clase por su prevalencia y renormalizar, $\tilde p_c \propto p_c / \pi_c$: la regla de Bayes con clases equiprobables.

In [3]:
import inspect  # Código fuente
print(inspect.getsource(mo.construir))  # El Pipeline de cada combinación

def construir(tarea, nombre, params, balanceo="Sin balanceo", semilla=42):
    """Construye el Pipeline completo de una combinación (preprocesamiento + balanceo + modelo).

    Parameters
    ----------
    tarea : {"clf", "reg"}
        Clasificación o regresión.
    nombre : str
        Modelo (uno de MODELOS_CLF o MODELOS_REG).
    params : dict
        Hiperparámetros del modelo.
    balanceo : str
        Estrategia de balanceo (solo clasificación).
    semilla : int
        Semilla para todos los componentes aleatorios.

    Returns
    -------
    imblearn.pipeline.Pipeline
        Pipeline sin ajustar.
    """
    pasos = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]  # Preprocesamiento (se ajusta en cada pliegue)
    if tarea == "clf":  # Clasificación
        if balanceo == "SMOTE":  # Interpolación entre vecinos de la clase minoritaria
            pasos.append(("sampler", SMOTE(k_neighbors=5, random_state=semilla)))
        elif balanceo == "A

## Validación cruzada anidada

Reportar la métrica del mejor modelo sobre el mismo esquema que lo eligió es optimista (sesgo de selección). Por eso hay dos bucles, ambos **por años y con purga de 4 semanas** (el objetivo de la fila $t$ está en $t+4$):

* **Bucle externo (3 pliegues):** cada año $a \in \{2019, 2020, 2021\}$ se evalúa con un modelo entrenado solo con los años anteriores. Es la métrica que se reporta (media ± desviación de los 3 pliegues).
* **Bucle interno (2 pliegues):** dentro del entrenamiento de cada pliegue externo, los dos últimos años validan; el optimizador solo ve el promedio de estos pliegues.

**¿Por qué 3 y 2?** El tiempo no se puede barajar: los pliegues son años completos, y un pliegue externo necesita al menos 4 años previos (2 para entrenar y 2 para validar dentro). 2019–2021 cubre un año epidémico (2019), uno de pandemia (2020) y uno de baja transmisión (2021), así que la desviación entre pliegues refleja regímenes reales. Para que las 420 tareas quepan en el presupuesto, el entrenamiento externo se submuestrea a 20 000 filas estratificadas por año y clase; el año evaluado se usa **completo**.

In [4]:
tr, va, te = dd.cargar()  # Periodos
filas = []  # Línea de tiempo de los pliegues
for a in c3.ANIOS_EXTERNOS:  # Pliegues externos
    ent, ev = dd.pliegue_externo(tr, a)  # Externo
    sub = dd.submuestra(ent, c3.N_EXTERNO)  # Submuestra
    for k, (i_tr, i_va) in enumerate(dd.pliegues_internos(sub), 1):  # Internos
        filas.append({"pliegue externo": a, "pliegue interno": k, "entrena (años)": f"{sub['ANO'].iloc[i_tr].min()}–{sub['ANO'].iloc[i_tr].max()}",
                      "filas internas de entrenamiento": len(i_tr), "valida (año)": int(sub["ANO"].iloc[i_va].iloc[0]), "filas internas de validación": len(i_va)})
    filas.append({"pliegue externo": a, "pliegue interno": "externo", "entrena (años)": f"{ent['ANO'].min()}–{a - 1}", "filas internas de entrenamiento": len(sub),
                  "valida (año)": a, "filas internas de validación": len(ev)})
pd.DataFrame(filas)

,pliegue externo,pliegue interno,entrena (años),filas internas de entrenamiento,valida (año),filas internas de validación
0,2019,1,2015–2016,9399,2017,4915
1,2019,2,2015–2017,14711,2018,4898
2,2019,externo,2015–2018,20000,2019,18048
3,2020,1,2015–2017,11632,2018,3872
4,2020,2,2015–2018,15807,2019,3871
5,2020,externo,2015–2019,20000,2020,18424
6,2021,1,2015–2018,13021,2019,3184
7,2021,2,2015–2019,16473,2020,3256
8,2021,externo,2015–2020,20000,2021,18048


In [5]:
fig = go.Figure()  # Diagrama de la validación anidada
y0 = 0  # Posición vertical
for a in c3.ANIOS_EXTERNOS:  # Pliegues externos
    for k, (ini_val) in enumerate([a - 2, a - 1]):  # Internos
        fig.add_bar(x=[ini_val - 2015], base=[2015], y=[f"externo {a} · interno {k + 1}"], orientation="h", marker_color=AZUL, showlegend=False)  # Entrena interno
        fig.add_bar(x=[1], base=[ini_val], y=[f"externo {a} · interno {k + 1}"], orientation="h", marker_color=AMBAR, showlegend=False)  # Valida interno
    fig.add_bar(x=[a - 2015], base=[2015], y=[f"externo {a}"], orientation="h", marker_color="#1e3a8a", showlegend=False)  # Entrena externo
    fig.add_bar(x=[1], base=[a], y=[f"externo {a}"], orientation="h", marker_color=ROJO, showlegend=False)  # Evalúa externo
fig.add_bar(x=[7], base=[2015], y=["reajuste final"], orientation="h", marker_color="#1e3a8a", showlegend=False)  # Final
fig.add_bar(x=[1], base=[2022], y=["reajuste final"], orientation="h", marker_color=VERDE, showlegend=False)  # Umbral y calibración
fig.add_bar(x=[2], base=[2023], y=["reajuste final"], orientation="h", marker_color=GRIS, showlegend=False)  # Prueba
fig.update_layout(barmode="overlay", title="Validación anidada: azul = entrena, ámbar = valida (interno), rojo = evalúa (externo), verde = 2022, gris = prueba",
                  xaxis=dict(dtick=1, title="año"), yaxis=dict(autorange="reversed"))
mostrar(fig, 520)  # Muestra

**Reajuste final.** Con los hiperparámetros elegidos en el pliegue externo más reciente (2021), cada combinación se ajusta sobre 2015–2021 (40 000 filas). La validación 2022 fija el umbral de decisión y la recalibración, y la prueba 2023–2024 se usa una sola vez para las métricas de los capítulos de resultados y para las pruebas estadísticas.

## Métrica de validación

El criterio que maximizan los cuatro optimizadores es el **AUC-PR** (precisión promedio) en clasificación y el **−RMSE** en regresión. Un brote no anticipado (falso negativo) cuesta más que una falsa alarma, pero la capacidad de respuesta de una secretaría también es limitada: el AUC-PR resume el compromiso precisión–recall en todos los umbrales y, a diferencia del AUC-ROC, se concentra en la clase brote y no se infla con los verdaderos negativos (90 % de las filas). El umbral concreto se elige después maximizando F1 en 2022; con él se reportan Accuracy, Precision, Recall y F1.

## Los cuatro optimizadores (y uno adicional)

| Método | Cómo propone el siguiente candidato | Exploración / explotación |
|---|---|---|
| **Grid Search** | Malla regular: la mayor que cabe en 20 evaluaciones (las dimensiones sin niveles quedan en el centro del rango) | Solo exploración, fija de antemano; sufre con muchas dimensiones |
| **Random Search** | Muestreo independiente, uniforme o log-uniforme | Solo exploración; cubre mejor las dimensiones importantes que la malla (Bergstra y Bengio, 2012) |
| **Bayesiana (Optuna, TPE)** | Modelo sustituto *Tree-structured Parzen Estimator*: estima $l(x) = p(x \mid y \text{ bueno})$ y $g(x) = p(x \mid y \text{ malo})$ con el cuantil γ de los mejores resultados, y propone el candidato que maximiza $l(x)/g(x)$ | La razón $l/g$ es equivalente a maximizar la **Mejora Esperada (EI)**; 5 evaluaciones aleatorias iniciales exploran y después explota, con exploración residual por los 24 candidatos muestreados de $l(x)$ |
| **Genética (DEAP)** | Población de 6 configuraciones codificadas en $[0, 1]^d$; selección por **torneo** (3), **cruce** de mezcla (α = 0,5, p = 0,7), **mutación** gaussiana (σ = 0,15, p = 0,3); estrategia (μ + λ) con **elitismo** | El cruce explota combinaciones de buenos padres; la mutación y los inmigrantes mantienen la **diversidad**, que se registra por generación |
| **Successive Halving** (complementario) | 27 candidatos con 1/9 de las filas, los mejores 9 con 1/3 y los mejores 3 con todas (factor 3) | Gasta poco en las configuraciones malas y reasigna el presupuesto a las competitivas |

**Sin fuga de datos en ningún método.** Optuna y DEAP no integran validación cruzada: la función objetivo implementada a mano ajusta el **Pipeline completo** (imputación, estandarización, remuestreo y modelo) dentro de cada pliegue interno, con los datos de entrenamiento del pliegue.

In [6]:
print(inspect.getsource(op.bayesiana))  # Optuna con TPE
print(inspect.getsource(op.genetica))  # DEAP

def bayesiana(espacio, objetivo, presupuesto, semilla=42, arranque=5):
    """Optimización bayesiana con Optuna: modelo sustituto TPE (Tree-structured Parzen Estimator).

    TPE modela p(x | y bueno) = l(x) y p(x | y malo) = g(x) con estimadores de densidad de Parzen y propone el candidato
    que maximiza l(x)/g(x), que es equivalente a maximizar la Mejora Esperada (EI). Las ``arranque`` primeras
    evaluaciones son aleatorias (exploración); después TPE explota las regiones prometedoras.
    """
    historial, t0 = [], time.perf_counter()  # Historial y cronómetro

    def f(trial):  # Función objetivo de Optuna
        p = {}  # Hiperparámetros sugeridos
        for k, spec in espacio.items():  # Cada dimensión
            if spec[0] == "cat":  # Categórica
                p[k] = trial.suggest_categorical(k, spec[1])
            elif spec[0] == "int":  # Entera
                p[k] = trial.suggest_int(k, spec[1], spec[2], log=spec[3])
            else:  # Real
                p[k] 

In [7]:
print(inspect.getsource(ex.ejecutar_tarea))  # Una tarea de la validación anidada

def ejecutar_tarea(tarea, modelo, balanceo, metodo, anio, semilla=c3.SEMILLA, presupuesto=c3.PRESUPUESTO, guardar=True, semilla_optimizador=None):
    """Ejecuta un pliegue externo de la validación anidada para una combinación.

    Parameters
    ----------
    tarea : {"clf", "reg"}
        Clasificación o regresión.
    modelo, balanceo, metodo : str
        Combinación (balanceo = "—" en regresión).
    anio : int
        Año del pliegue externo.
    semilla : int
        Semilla del optimizador, del muestreo y de los modelos.
    presupuesto : int
        Evaluaciones de hiperparámetros.
    guardar : bool
        Si True, guarda el resultado en resultados/tareas.
    semilla_optimizador : int, optional
        Semilla solo del optimizador (los datos y los modelos conservan ``semilla``); sirve para aislar la variabilidad del
        método de búsqueda en el estudio de estabilidad.

    Returns
    -------
    dict
        Registro con los mejores hiperparámetros, las métricas del 

## Tabla maestra de experimentos

Cada una de las 420 tareas (140 combinaciones × 3 pliegues externos) se registra con su modelo, balanceo, optimizador, semilla, hiperparámetros finales, métricas del bucle externo, número de evaluaciones y tiempos. Se guarda en `resultados/tabla_maestra.csv`.

In [8]:
tabla = pd.read_csv(c3.RESULTADOS / "tabla_maestra.csv")  # Tabla maestra
print(f"Registros: {len(tabla)} · combinaciones: {tabla.groupby(['tarea', 'modelo', 'balanceo', 'optimizador']).ngroups} · "
      f"evaluaciones de hiperparámetros: {int(tabla['evaluaciones'].sum()):,} · ajustes de modelos en el bucle interno: {int(tabla['evaluaciones'].sum()) * c3.PLIEGUES_INTERNOS:,}")
tabla.drop(columns=["mejores_params"]).head(8)

Registros: 420 · combinaciones: 140 · evaluaciones de hiperparámetros: 8,250 · ajustes de modelos en el bucle interno: 16,500


,tarea,modelo,balanceo,optimizador,anio_externo,semilla,valor_interno,AUC-ROC,AUC-PR,Brier,evaluaciones,segundos_optimizacion,segundos_por_evaluacion,segundos_ajuste,segundos_prediccion_1000,n_entrenamiento,n_evaluacion,errores,RMSE,MAE,R2
0,clf,KNN,ADASYN,Bayesiana (Optuna TPE),2019,42,0.3783,0.8936,0.7459,0.1936,20,63.9362,3.1968,0.5337,0.4393,20000,18048,0,NaN,NaN,NaN
1,clf,KNN,ADASYN,Bayesiana (Optuna TPE),2020,42,0.6812,0.8949,0.5972,0.1821,20,58.6780,2.9339,0.5852,0.4682,20000,18424,0,NaN,NaN,NaN
2,clf,KNN,ADASYN,Bayesiana (Optuna TPE),2021,42,0.6601,0.9041,0.5400,0.1029,20,54.2040,2.7102,0.4740,0.3802,20000,18048,0,NaN,NaN,NaN
3,clf,KNN,ADASYN,Genética (DEAP),2019,42,0.3746,0.8932,0.7422,0.1944,20,55.6968,2.7848,0.3457,0.3817,20000,18048,0,NaN,NaN,NaN
4,clf,KNN,ADASYN,Genética (DEAP),2020,42,0.6783,0.8941,0.5856,0.1838,20,49.7066,2.4853,0.4544,0.3441,20000,18424,0,NaN,NaN,NaN
5,clf,KNN,ADASYN,Genética (DEAP),2021,42,0.6533,0.9048,0.5352,0.1039,20,46.7871,2.3394,0.4441,0.3903,20000,18048,0,NaN,NaN,NaN
6,clf,KNN,ADASYN,Grid Search,2019,42,0.3779,0.8925,0.7436,0.1952,20,55.1898,2.7595,0.4198,0.4858,20000,18048,0,NaN,NaN,NaN
7,clf,KNN,ADASYN,Grid Search,2020,42,0.6827,0.8947,0.5961,0.1828,20,51.2890,2.5645,0.6048,0.4718,20000,18424,0,NaN,NaN,NaN


**Reproducibilidad.** Una sola semilla global (`SEMILLA = 42`, `e3/config.py`) se propaga a NumPy, scikit-learn, XGBoost, imbalanced-learn, Optuna (`TPESampler(seed)`) y DEAP (`random.seed`). Las versiones exactas están en `entregable3/requirements.txt`. El código está separado en módulos (datos, modelos, optimizadores, experimento, evaluación, estadística, complejidad) con *docstrings* estilo NumPy en las funciones compartidas.